In [0]:
-- Normal Unity Catalog managed Delta table
CREATE TABLE catalog.default.managed_plain (
  id BIGINT,
  batch_id INT,
  category STRING,
  amount DECIMAL(12,2),
  event_ts TIMESTAMP,
  event_date DATE
);

-- Same managed table, but with catalog commits enabled
CREATE TABLE catalog_commits.default.managed_catalog_commits (
  id BIGINT,
  batch_id INT,
  category STRING,
  amount DECIMAL(12,2),
  event_ts TIMESTAMP,
  event_date DATE
)
TBLPROPERTIES (
  'delta.feature.catalogManaged' = 'supported'
);

In [0]:
%python
       
# Cell 2: create many small commits in both tables

tables = [
    "catalog.default.managed_plain",
    "catalog_commits.default.managed_catalog_commits"
]

# Many tiny inserts = many Delta history versions
for i in range(1, 100):
    for t in tables:
        spark.sql(f"""
            INSERT INTO {t}
            SELECT
              id,
              batch_id,
              category,
              amount,
              event_ts,
              event_date,
              comment
            FROM (
              SELECT
                {i} * 100 + id AS id,
                {i} * 100 + id AS batch_id,
                CASE WHEN id % 2 = 0 THEN 'even' ELSE 'odd' END AS category,
                CAST(({i} * 10.25 + id) AS DECIMAL(12,2)) AS amount,
                current_timestamp() AS event_ts,
                DATE '2026-05-13' AS event_date,
                'comment' AS comment
              FROM RANGE(0, 100)
            )
        """)

In [0]:
%python
# Cell 3: add more diverse history events

for t in tables:
    # UPDATE event
    spark.sql(f"""
        UPDATE {t}
        SET amount = amount + 1
        WHERE id IN (1, 2, 3)
    """)

    # DELETE event
    spark.sql(f"""
        DELETE FROM {t}
        WHERE id = 4
    """)

    # MERGE event
    spark.sql(f"""
        MERGE INTO {t} AS target
        USING (
          SELECT 2 AS id, 999 AS batch_id, 'merged' AS category,
                 CAST(222.22 AS DECIMAL(12,2)) AS amount,
                 current_timestamp() AS event_ts,
                 DATE '2026-05-13' AS event_date
        ) AS source
        ON target.id = source.id
        WHEN MATCHED THEN UPDATE SET
          target.batch_id = source.batch_id,
          target.category = source.category,
          target.amount = source.amount
        WHEN NOT MATCHED THEN INSERT *
    """)

    # Table property change event
    spark.sql(f"""
        ALTER TABLE {t}
        SET TBLPROPERTIES (
          'demo.history_event' = 'property_update_1'
        )
    """)

    # Add column event
    spark.sql(f"""
        ALTER TABLE {t}
        ADD COLUMNS (comment STRING)
    """)

    # Insert after schema change
    spark.sql(f"""
        INSERT INTO {t}
        VALUES (
          1001,
          1001,
          'after_schema_change',
          CAST(1001.01 AS DECIMAL(12,2)),
          current_timestamp(),
          DATE '2026-05-14',
          'new column populated'
        )
    """)